### Load the Data

In [ ]:
from pathlib import Path
import pandas as pd
PROJECT_ROOT = next(
  p for p in [Path.cwd(), *Path.cwd().parents] if (p / "requirements.txt").exists()
)

RAW_DIR = PROJECT_ROOT / "data" / "raw"

def load(name: str) -> pd.DataFrame():
  df = pd.read_csv(RAW_DIR / f"{name}.csv", dtype=str, keep_default_na=False)
  df.columns = df.columns.str.strip().str.lower()
  return df

items = load("order_items")
options = load("order_item_options")
dates = load("date_dim")
print(items.shape, options.shape, dates.shape)


(203519, 13) (193017, 6) (365, 8)
                   order_id               lineitem_id option_group_name  \
0  6278eb58dfafab1e053fe172  6278eb63dfafab1e053fe18c            Cheese   
1  6278eb58dfafab1e053fe172  6278eb63dfafab1e053fe18c           Veggies   
2  6278eb58dfafab1e053fe172  6278eb63dfafab1e053fe18c           Veggies   
3  6278eb58dfafab1e053fe172  6278eb63dfafab1e053fe18c           Veggies   
4  6278eb58dfafab1e053fe172  6278eb63dfafab1e053fe18c           Veggies   

       option_name option_price option_quantity  
0  American Cheese            0               1  
1          Lettuce          0.5               1  
2         Tomatoes            0               1  
3            Onion            0               1  
4    Green Peppers            0               1  


## Distinct values per column

In [5]:
tables = {"order_items": items, "order_item_options": options, "date_dim": dates}

for table_name, df in tables.items():
  print(f"\n====== {table_name} ======")
  for col in df.columns:
      distinct = df[col].nunique()
      top5 = df[col].value_counts().head(5).to_dict()
      print(f"{col:<22} distinct={distinct:<8} top5={top5}")


====== order_items ======
app_name               distinct=3        top5={'Alltown Fresh': 201423, 'Alltown Neighborhood Perks': 1270, 'Alltown Fresh - DEVELOPMENT': 826}
restaurant_id          distinct=28       top5={'5fece84aadb0d51509c36f22': 32459, '6054db3495b70198148b456f': 23135, '5e7e35ec902ad5ac017b242a': 21416, '5f6a6c1537ab46bd38e9df75': 18888, '62f2ce9824813746ce6f5140': 12571}
creation_time_utc      distinct=131328   top5={'2021-02-08T12:16:06.916Z': 61, '2020-12-09T15:11:43.976Z': 27, '2022-05-03T14:14:56.527Z': 26, '2021-12-07T16:23:30.917Z': 25, '2022-04-27T13:51:27.864Z': 23}
order_id               distinct=131328   top5={'60212b86505ee90c1e92ec74': 61, '5fd0e92f4f5ee9301fda8df8': 27, '627138e08ae3be97e9047d77': 26, '61af8a82ffd125453e62ae37': 25, '62694a5fcb0036591d459146': 23}
user_id                distinct=20175    top5={'': 17808, '5f1b00e5535ee93e0cb768e7': 2538, '5ece77fe902ad501337b23fd': 2454, '609d681462e498b356e72a6d': 1133, '5eac88d6902ad598127b240b': 1046}

## Numeric ranges and two confirmations

In [17]:
item_nums = items[["item_price", "item_quantity"]].apply(pd.to_numeric)
print(item_nums.describe())
print("item_price <=0   :", (item_nums["item_price"]<=0).sum())
print("item_quantity <=0:", (item_nums["item_quantity"]<=0).sum())

option_price = pd.to_numeric(options["option_price"])
print("\nall option_price values:", sorted(option_price.unique().tolist()))
print("negative option_price rows:", (option_price<0).sum())

print()
print(pd.crosstab(
  items["is_loyalty"],
  items["printed_card_number"]=="",
  rownames=["is_loyalty"],
  colnames=["card_blank"]
))

          item_price  item_quantity
count  203519.000000  203519.000000
mean        8.739889       1.117768
std        19.568335       2.313234
min         0.000000       0.000000
25%         5.890000       1.000000
50%         8.000000       1.000000
75%        10.000000       1.000000
max      5000.000000     500.000000
item_price <=0   : 156
item_quantity <=0: 1

all option_price values: [0.0, 0.05, 0.25, 0.5, 0.6, 0.75, 1.0, 1.25, 1.5, 1.99, 2.0, 2.5, 3.0, 4.0, 6.0, 8.0]
negative option_price rows: 0

card_blank  False   True 
is_loyalty               
FALSE           0  157435
TRUE        46084       0


## Inspect the outliers

In [41]:
items_n = items.assign(
    item_price=pd.to_numeric(items["item_price"]),
    item_quantity=pd.to_numeric(items["item_quantity"]),
)
items_n["line_total"] = items_n["item_price"] * items_n["item_quantity"]

cols = ["app_name", "order_id", "user_id", "item_name", "item_price", "item_quantity", "line_total"]

print("Top 10 line totals:")
print(items_n.nlargest(10, "line_total")[cols].to_string())

print("\nZero-price items (top 10 by count):")
print(items_n.loc[items_n["item_price"] == 0, "item_name"].value_counts().head(10))

print("\nZero-quantity row:")
print(items_n.loc[items_n["item_quantity"] == 0, cols].to_string())

Top 10 line totals:
             app_name                  order_id                   user_id                   item_name  item_price  item_quantity  line_total
118704  Alltown Fresh  5f3d1ba7505ee9374b7b23cc  5ee7285f505ee9f7490cf93d               Korean Kimchi     5000.00            500  2500000.00
111543  Alltown Fresh  5ed7972a4f5ee90547011c03  5ecf9eda505ee9682b445912     Meet Your Matcha - 12oz     3500.00            500  1750000.00
126586  Alltown Fresh  615f4c9662e4983d105aeaa4                                    Chili Chicken Bowl     3567.00            300  1070100.00
161049  Alltown Fresh  615ca07661e498d660e720b9                                               B.L.A.T     2967.00            300   890100.00
111542  Alltown Fresh  5ed7972a4f5ee90547011c03  5ecf9eda505ee9682b445912           Espresso - Double     1125.00            500   562500.00
165682  Alltown Fresh  5ecf5b9b505ee9613c7b23dd  5ecf53924f5ee9de377b23ca                  Tuscan Sun     1917.00            213   408

## Test the hypothesis on every multi quantity row

In [ ]:
single = items_n[items_n["item_quantity"]==1] # filter down to the row with single quantity
reference_price = single.groupby("item_name")['item_price'].median() # get the median reference price per item_name

multi = items_n[items_n["item_quantity"]>1].copy() # filter down to rows where quantity is greater than 1
multi["reference_price"] = multi["item_name"].map(reference_price) # mapp the reference price to the multi quantity item
multi["price_per_unit"] = multi["item_price"] / multi["item_quantity"] # get the price for each unit when quantity is higher than 1

looks_like_unit = (multi["item_price"] - multi["reference_price"]).abs() < 0.01 # check if the item_price same as the singular item price
looks_like_total = (multi["price_per_unit"] - multi["reference_price"]).abs() < 0.01 # checks if the calculated price per unit same as refernce price from singular item

print("multi-quantity rows      :", len(multi))
print("item_price = unit price  :", looks_like_unit.sum())
print("item_price = line total  :", looks_like_total.sum())
print("neither                  :", (~looks_like_unit & ~looks_like_total).sum())

multi-quantity rows      : 13650
item_price = unit price  : 0
item_price = line total  : 5972
neither                  : 7678


## Check the "neither" rows

In [48]:
neither = multi[~looks_like_unit & ~looks_like_total]
print("no reference price:", neither["reference_price"].isna().sum())
ratio = (neither["price_per_unit"] / neither["reference_price"]).dropna()
print(ratio.describe())

no reference price: 16
count    7662.000000
mean        1.023115
std         0.222109
min         0.000000
25%         0.831791
50%         0.983306
75%         1.150250
max         3.428571
dtype: float64


## How are options charged?
The data has no order-total column, so you can't check option pricing directly against what the customer paid. But there's an indirect test, and it ties back to finding 9.

The idea: since option_quantity is always 1, an option bought twice must appear as two rows. So check whether multi-quantity line items have their options repeated once per unit. For example, does a line with quantity 3 have "Add Bacon" recorded 3 times?

If so, options are recorded per unit, and the 2,299 "duplicates" are real per-unit purchases.
If options appear once per line regardless of quantity, they're recorded per line.

In [59]:
option_counts = (
  options.groupby(["order_id","lineitem_id","option_group_name","option_name"])
  .size()
  .reset_index(name="times_recorded")
)

line_quantity = items_n[["order_id","lineitem_id", "item_quantity"]]
merged = option_counts.merge(line_quantity, on=["order_id","lineitem_id"], how="left")

print(merged.head(10).to_string())
print("option with no matching line item:", merged["item_quantity"].isna().sum())
print()
print(pd.crosstab(
  merged["item_quantity"].clip(upper=5),
  merged["times_recorded"].clip(upper=5),
  rownames=["item_quantity"],
  colnames=["times_recorded"]
))


                   order_id               lineitem_id    option_group_name    option_name  times_recorded  item_quantity
0  5e9f41ec505ee9c65088f162  5e9f41f4902ad5e0529516c1           Condiments        Ketchup               1            1.0
1  5e9f42aa4f5ee95e12c276b0  5e9f42b0902ad517529516f0           Condiments        Mustard               1            1.0
2  5eab19e8902ad559487b23d7  5eab19f4902ad500497b23ee           Condiments        Mustard               1            1.0
3  5eab1b3f902ad58e4d7b23ca  5eab1b44902ad5ce467b2478           Condiments        Mustard               1            1.0
4  5eb046fb902ad54903c80673  5eb04712902ad58e02c80655           Condiments        Ketchup               1            1.0
5  5eb046fb902ad54903c80673  5eb04712902ad58e02c80655           Condiments        Mustard               1            1.0
6  5eb04b7b505ee98e784837e2  5eb04b87902ad5f80dc8069d           Condiments        Ketchup               1            1.0
7  5eb16e26902ad59a237b2404  5eb

## How much the option-pricing question matters

In [67]:
opts = options.assign(option_price=pd.to_numeric(options['option_price']))
opts = opts.merge(line_quantity, on=["order_id", "lineitem_id"], how="inner")

item_revenue = items_n["item_price"].sum()
options_once = opts["option_price"].sum()
options_per_unit = (opts["option_price"] * opts["item_quantity"]).sum()
difference = options_per_unit - option_once
print(f"item revenue               : ${item_revenue:,.2f}")
print(f"options, once per line     : ${options_once:,.2f}")
print(f"options, x item_quantity   : ${options_per_unit:,.2f}")
print(f"difference                 : ${difference:,.2f} "
      f"({difference / (item_revenue + options_once):.2%} of total revenue)")



item revenue               : $1,778,733.53
options, once per line     : $85,245.14
options, x item_quantity   : $97,700.79
difference                 : $12,455.65 (0.67% of total revenue)
